# 03 · Risk assessment and comparison

**Owner: Risk Scoring.** Min-max rankings are *relative and illustrative*, not empirical livestock losses or future climate predictions.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("LivestockWeatherNotebook").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
from livestock_weather.settings import load_settings
cfg = load_settings(ROOT / "config" / "demo.json")


In [ ]:
from livestock_weather.scoring import score_states, compare_states
summary_path = ROOT / "data" / "demo_processed" / "state_summary.parquet"
if not summary_path.exists():
    raise FileNotFoundError("Run 02_weather_analysis.ipynb first")
summary = spark.read.parquet(str(summary_path))
scores = score_states(summary, cfg)
scores.select("state", "heat_score", "cold_score", "rain_score", "overall_score", "rank").show(60, False)
print("Comparison: TX, IA, WI")
compare_states(scores, ["TX", "IA", "WI"]).select("state", "overall_score", "rank").show()

In [ ]:
from livestock_weather.visuals import plot_ranking
figure = plot_ranking(scores, ROOT / "reports" / "figures" / "demo_state_ranking.png")
from IPython.display import display, Image
display(Image(filename=str(figure)))
scores.write.mode("overwrite").parquet(str(ROOT / "data" / "demo_processed" / "state_ranking.parquet"))